# Activation Addition (ActAdd)

**Paper**: [Steering Language Models With Activation Engineering](https://arxiv.org/abs/2308.10248)

**Authors**: Alexander Matt Turner, Lisa Thiergart, Gavin Leech, David Udell, Juan J. Vazquez, Ulisse Mini, Monte MacDiarmid

Activation Addition (ActAdd) is a state control method that steers model behavior by computing a positional steering vector from a single pair of short prompts and adding it to the residual stream at a single layer.

## Method Parameters

| parameter              | type                | description                                                                                   |
| ---------------------- | ------------------- | --------------------------------------------------------------------------------------------- |
| `positive_prompt`      | `str`               | Prompt representing the desired direction (e.g., `"Love"`)                                    |
| `negative_prompt`      | `str`               | Prompt representing the opposite direction (e.g., `"Hate"`)                                   |
| `steering_vector`      | `SteeringVector`    | Pre-computed steering vector (alternative to prompts); must be extracted at the layer-input boundary |
| `layer_id`             | `int`               | Layer to inject at. If `None`, defaults to ~20% depth                                         |
| `multiplier`           | `float`             | Scaling coefficient (called `c` in the paper). Typical values range from 1 to 15              |
| `alignment`            | `int`               | Absolute token position at which injection begins (called `a` in the paper); row `t` of the vector is added at position `alignment + t`. Default: 0 |
| `normalize_vector`     | `bool`              | If `True`, L2-normalize each position's direction vector before applying                      |
| `use_norm_preservation`| `bool`              | If `True`, wrap the transform in `NormPreservingTransform` to prevent distribution shift      |

## Setup

If running this from a Google Colab notebook, please uncomment the following cell to install the toolkit. The following block is not necessary if running this notebook from a virtual environment where the package has already been installed.

In [1]:
# !git clone https://github.com/generative-computing/steerability.git
# %cd steerability

In [2]:
import re
import textwrap

import torch
from tabulate import tabulate
from transformers import AutoModelForCausalLM, AutoTokenizer

from steerability.algorithms.core.steering_pipeline import SteeringPipeline
from steerability.algorithms.state_control.act_add.control import ActAdd

For this demonstration, we use `Qwen2.5-1.5B`. Since ActAdd works with raw continuation prompts, we use the base model rather than the instruction-tuned variant. We load the model and tokenizer once and share them across the baseline and both steering pipelines.

In [3]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
device = model.device

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Baseline versus steered behavior for the sentiment example will be studied using the following test prompts.

In [4]:
test_prompts = [
    "I hate you because",
    "I think you're",
    "My favorite thing about life is",
    "I went up to my friend and said",
]

## Baseline Model Behavior

We first generate responses from the (unsteered) baseline model. Note that we reset the random seed before each generation so that the sampled completions are reproducible.

In [5]:
gen_params = {
    "max_new_tokens": 30,
    "do_sample": True,
    "temperature": 1.0,
    "top_p": 0.3,
    "repetition_penalty": 1.1,
    "pad_token_id": tokenizer.eos_token_id,
}

baseline_responses = []
for prompt in test_prompts:
    torch.manual_seed(0)
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(**enc, **gen_params)
    response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    baseline_responses.append(response)

print("Baseline completions:\n")
for prompt, response in zip(test_prompts, baseline_responses):
    print(f"Prompt: {prompt}")
    print(f"Response: {response}\n")

Baseline completions:

Prompt: I hate you because
Response: I hate you because I love you. I love you because I hate you. You are the only one who can make me feel this way. You are my everything,

Prompt: I think you're
Response: I think you're right. I'm not sure what to do about it, though.

Does it follow that "I don't know how to fix this."?
Choose

Prompt: My favorite thing about life is
Response: My favorite thing about life is the ability to make new friends. I love meeting people and making them feel welcome in my home. My mom was a very outgoing person, so she

Prompt: I went up to my friend and said
Response: I went up to my friend and said "I'm sorry I forgot about you" but he didn't understand what I meant. He thought I was talking about him.

Is it possible that



## Token alignment of the contrast pairs

ActAdd takes the per-position difference between the residual streams of the two prompts, which requires the prompts to be aligned token by token. Following Algorithm 1 of the paper, the estimator right-pads the shorter prompt with attended space tokens. The rows of the vector beyond the shorter prompt are therefore differences between a trailing space and the remaining tokens of the longer prompt. Under Qwen's tokenizer `Hate` is two tokens and `Love` is one, which gives the sentiment vector two rows (the second against a space). The wedding pair spans five and seven tokens, which places rows 5 and 6 of its vector against spaces.

In [6]:
contrast_pairs = [
    ("Love", "Hate"),
    ("I talk about weddings constantly", "I do not talk about weddings constantly"),
]
for positive, negative in contrast_pairs:
    positive_ids = tokenizer.encode(positive, add_special_tokens=False)
    negative_ids = tokenizer.encode(negative, add_special_tokens=False)
    print(f"{positive!r}: {len(positive_ids)} tokens; {negative!r}: {len(negative_ids)} tokens")

'Love': 1 tokens; 'Hate': 2 tokens
'I talk about weddings constantly': 5 tokens; 'I do not talk about weddings constantly': 7 tokens


## Sentiment Steering

As in the original paper, we demonstrate sentiment steering using a "Love" vs "Hate" prompt pair, applied at layer 8 with a multiplier of 8. The `alignment` argument gives the absolute token position where injection begins; row `t` of the steering vector is added at position `alignment + t`. Qwen's tokenizer prepends no special tokens to the prompt, so `alignment=0` places the steering vector on the first prompt tokens. The vector has two rows (see the token counts above) and therefore covers the first two prompt tokens.

In [7]:
act_add_sentiment = ActAdd(
    positive_prompt="Love",
    negative_prompt="Hate",
    layer_id=8,
    multiplier=8,
    alignment=0,
)

sentiment_pipeline = SteeringPipeline(
    model=model,
    tokenizer=tokenizer,
    controls=[act_add_sentiment],
)
sentiment_pipeline.steer()

In [8]:
sentiment_responses = []
for prompt in test_prompts:
    torch.manual_seed(0)
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    output_ids = sentiment_pipeline.generate(
        input_ids=enc.input_ids,
        attention_mask=enc.attention_mask,
        return_full_sequence=True,
        **gen_params,
    )
    response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    sentiment_responses.append(response)

def wrap(text, width=50):
    return '\n'.join(textwrap.wrap(text, width=width))

table_data = []
for i, prompt in enumerate(test_prompts):
    table_data.append([
        wrap(prompt, 20),
        wrap(baseline_responses[i], 40),
        wrap(sentiment_responses[i], 40),
    ])

print(tabulate(
    table_data,
    headers=["prompt", "baseline", "steered (Love - Hate)"],
    tablefmt="grid",
))

+--------------------+------------------------------------------+--------------------------------------------+
| prompt             | baseline                                 | steered (Love - Hate)                      |
+====================+==========================================+============================================+
| I hate you because | I hate you because I love you. I love    | I hate you because 1. I love 2. You love   |
|                    | you because I hate you. You are the only | me 3. You are a great person 4. I am       |
|                    | one who can make me feel this way. You   | your best friend 5.                        |
|                    | are my everything,                       |                                            |
+--------------------+------------------------------------------+--------------------------------------------+
| I think you're     | I think you're right. I'm not sure what  | I think you're right. I'll have to go      |
|

## Topic Steering

ActAdd can also be used to steer the model toward specific topics. We use the wedding example from the paper, intervening at layer 9 with `alignment=0` as before. The pair spans five and seven tokens, which gives the vector seven rows (rows 5 and 6 are differences against trailing spaces). The injection window therefore covers the first seven tokens of the sequence. The multiplier is chosen by the sweep in the next section. We evaluate on prompts of at least seven tokens, so that the whole window lies within the prompt, and generate baseline completions for these prompts first.

In [9]:
topic_prompts = [
    "I went up to my friend and said",
    "Yesterday my sister called to tell me about",
    "The best part of my weekend was when",
    "Last night at dinner my parents told us",
]

topic_baseline_responses = []
for prompt in topic_prompts:
    torch.manual_seed(0)
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(**enc, **gen_params)
    response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    topic_baseline_responses.append(response)

## Choosing the multiplier

The multiplier trades the strength of the topic shift against damage to the prompt representation at the padded positions, which shows up as doubled spaces or full-width punctuation in the continuation. We sweep the multiplier at layer 9 and sample eight completions per prompt (seeds 0 to 7) for each value. We then count the completions that mention a wedding-related word and the completions that show those artifacts. The unsteered model gives the reference rate.

In [10]:
MULTIPLIERS = [2, 4, 6, 8]
N_SEEDS = 8
WEDDING_WORDS = re.compile(r"\b(wedding|weddings|bride|groom|marr(?:y|ied|iage)|honeymoon|vows?)\b", re.IGNORECASE)
ARTIFACTS = re.compile(r"[\u3000-\u303F\uFF00-\uFFEF]|  ")  # full-width punctuation or a doubled space

def topic_rates(pipeline):
    mentions = artifacts = 0
    for prompt in topic_prompts:
        enc = tokenizer(prompt, return_tensors="pt").to(device)
        for seed in range(N_SEEDS):
            torch.manual_seed(seed)
            output_ids = pipeline.generate(input_ids=enc.input_ids, attention_mask=enc.attention_mask, **gen_params)
            text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
            mentions += bool(WEDDING_WORDS.search(text))
            artifacts += bool(ARTIFACTS.search(text))
    total = len(topic_prompts) * N_SEEDS
    return mentions / total, artifacts / total

unsteered = SteeringPipeline(model=model, tokenizer=tokenizer, controls=[])
unsteered.steer()
sweep_rows = [["unsteered", *topic_rates(unsteered)]]
for multiplier in MULTIPLIERS:
    control = ActAdd(
        positive_prompt="I talk about weddings constantly",
        negative_prompt="I do not talk about weddings constantly",
        layer_id=9,
        multiplier=multiplier,
        alignment=0,
    )
    pipeline = SteeringPipeline(model=model, tokenizer=tokenizer, controls=[control])
    pipeline.steer()
    sweep_rows.append([f"multiplier = {multiplier}", *topic_rates(pipeline)])

print(tabulate(sweep_rows, headers=["configuration", "wedding mentions", "artifacts"], floatfmt=".2f", tablefmt="grid"))

+-----------------+--------------------+-------------+
| configuration   |   wedding mentions |   artifacts |
+=================+====================+=============+
| unsteered       |               0.03 |        0.03 |
+-----------------+--------------------+-------------+
| multiplier = 2  |               0.19 |        0.47 |
+-----------------+--------------------+-------------+
| multiplier = 4  |               0.19 |        1.00 |
+-----------------+--------------------+-------------+
| multiplier = 6  |               0.31 |        1.00 |
+-----------------+--------------------+-------------+
| multiplier = 8  |               0.38 |        1.00 |
+-----------------+--------------------+-------------+


We select the smallest multiplier whose wedding rate is at least twice the unsteered rate and whose artifact rate is at most 0.10. When no multiplier meets both conditions, we select the multiplier with the highest wedding rate. In this run every multiplier raises the wedding rate above twice the unsteered rate (0.03), but none keeps the artifact rate at or below 0.10 (the lowest is 0.47, at multiplier 2). The rule therefore selects multiplier 8, which has a wedding rate of 0.38 and artifacts in every completion.

In [11]:
TOPIC_MULTIPLIER = 8

In [12]:
act_add_topic = ActAdd(
    positive_prompt="I talk about weddings constantly",
    negative_prompt="I do not talk about weddings constantly",
    layer_id=9,
    multiplier=TOPIC_MULTIPLIER,
    alignment=0,
)

topic_pipeline = SteeringPipeline(
    model=model,
    tokenizer=tokenizer,
    controls=[act_add_topic],
)
topic_pipeline.steer()

In [13]:
topic_responses = []
for prompt in topic_prompts:
    torch.manual_seed(0)
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    output_ids = topic_pipeline.generate(
        input_ids=enc.input_ids,
        attention_mask=enc.attention_mask,
        return_full_sequence=True,
        **gen_params,
    )
    response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    topic_responses.append(response)

table_data = []
for i, prompt in enumerate(topic_prompts):
    table_data.append([
        wrap(prompt, 20),
        wrap(topic_baseline_responses[i], 40),
        wrap(topic_responses[i], 40),
    ])

print(tabulate(
    table_data,
    headers=["prompt", "baseline", "steered (weddings)"],
    tablefmt="grid",
))

+----------------------+------------------------------------------+--------------------------------------------+
| prompt               | baseline                                 | steered (weddings)                         |
+======================+==========================================+============================================+
| I went up to my      | I went up to my friend and said "I'm     | I went up to my friend and said，  "I'm    |
| friend and said      | sorry I forgot about you" but he didn't  | going to be a wedding photographer．"  I   |
|                      | understand what I meant. He thought I    | had no idea what I was doing at the        |
|                      | was talking about him.  Is it possible   | time．I just knew that                     |
|                      | that                                     |                                            |
+----------------------+------------------------------------------+--------------------------------

## Summary

This notebook demonstrated Activation Addition (ActAdd) for lightweight behavior steering:

1. ActAdd computes a positional steering vector from two short prompts.
2. The sentiment example showed how a simple "Love" vs "Hate" contrast shifts emotional tone.
3. The topic example raised the share of completions that mention weddings from 0.03 (unsteered) to 0.38 at multiplier 8. Every steered completion at that strength contains full-width punctuation or doubled spaces, and the smallest multiplier in the sweep (2) shows these artifacts at a rate of 0.47.

ActAdd trades off statistical robustness (using more than a single prompt pair) for speed and simplicity, compared to contrastive activation addition (CAA) which aggregates over many pairs. The positional nature of the steering vector (row `t` is added at absolute position `alignment + t`, rather than broadcasting one vector over all positions) allows fine-grained control over where in the sequence the steering takes effect. The estimator right-pads the shorter prompt of a pair with attended space tokens. A pair of unequal token length therefore produces rows that compare a trailing space with the tokens of the longer prompt, and pairs of equal length have no such rows.